In [13]:
from ultralytics import YOLO

# Load model
model = YOLO("/home/debasish/Documents/YOLOv8/weights/pose/yolov8l-pose-1024.pt")

# Perform tracking on video or image sequence
results = model.track(source="/home/debasish/Documents/YOLOv8/videos/new_test_videos/mota/videos/video7.mp4", 
                      save=True, tracker="botsort.yaml",stream=True)

# Save results in MOTChallenge format (frame, id, bbox, conf)
with open('track_video7.txt', 'w') as f:
    for frame_id, result in enumerate(results):
        for box in result.boxes:
            bbox = box.xyxy[0].tolist()  # Convert from tensor to list
            track_id = box.id.item()     # Get track id
            conf = box.conf.item()       # Get confidence score
            f.write(f'{frame_id+1},{track_id},{bbox[0]},{bbox[1]},{bbox[2]-bbox[0]},{bbox[3]-bbox[1]},-1,-1,{conf}\n')


video 1/1 (frame 1/365) /home/debasish/Documents/YOLOv8/videos/new_test_videos/mota/videos/video7.mp4: 576x1024 16 worms, 17.8ms
video 1/1 (frame 2/365) /home/debasish/Documents/YOLOv8/videos/new_test_videos/mota/videos/video7.mp4: 576x1024 15 worms, 17.8ms
video 1/1 (frame 3/365) /home/debasish/Documents/YOLOv8/videos/new_test_videos/mota/videos/video7.mp4: 576x1024 15 worms, 18.8ms
video 1/1 (frame 4/365) /home/debasish/Documents/YOLOv8/videos/new_test_videos/mota/videos/video7.mp4: 576x1024 16 worms, 18.8ms
video 1/1 (frame 5/365) /home/debasish/Documents/YOLOv8/videos/new_test_videos/mota/videos/video7.mp4: 576x1024 16 worms, 16.3ms
video 1/1 (frame 6/365) /home/debasish/Documents/YOLOv8/videos/new_test_videos/mota/videos/video7.mp4: 576x1024 16 worms, 15.0ms
video 1/1 (frame 7/365) /home/debasish/Documents/YOLOv8/videos/new_test_videos/mota/videos/video7.mp4: 576x1024 16 worms, 15.3ms
video 1/1 (frame 8/365) /home/debasish/Documents/YOLOv8/videos/new_test_videos/mota/videos/video

In [1]:
import numpy as np
import pandas as pd
import motmetrics as mm
from pathlib import Path

# Load MOT files (they have NO header)
cols = [
    "FrameId", "Id", "X", "Y", "Width", "Height",
    "Confidence", "ClassId", "Visibility"
]

gt_df   = pd.read_csv("./gt/video7_gt.txt", header=None, names=cols)
pred_df = pd.read_csv("track_video7.txt", header=None, names=cols)

# make sure FrameId and Id are integers
for df in (gt_df, pred_df):
    df["FrameId"] = df["FrameId"].astype(int)
    df["Id"]      = df["Id"].astype(int)


# put into the MultiIndex format motmetrics expects
gt_df   = gt_df.set_index(["FrameId", "Id"])
pred_df = pred_df.set_index(["FrameId", "Id"])

# build MOTAccumulator by looping over frames
acc = mm.MOTAccumulator(auto_id=True)

all_frames = sorted(
    set(gt_df.index.get_level_values(0)) |
    set(pred_df.index.get_level_values(0))
)

iou_threshold = 0.5  # tunable

for frame in all_frames:
    # ground truth objects at this frame
    gt_f = gt_df.xs(frame, level=0, drop_level=False) \
        if frame in gt_df.index.get_level_values(0) else None
    # predicted objects at this frame
    pr_f = pred_df.xs(frame, level=0, drop_level=False) \
        if frame in pred_df.index.get_level_values(0) else None

    gt_ids = [] if gt_f is None else gt_f.index.get_level_values(1).tolist()
    pr_ids = [] if pr_f is None else pr_f.index.get_level_values(1).tolist()

    gt_boxes = [] if gt_f is None else gt_f[["X", "Y", "Width", "Height"]].to_numpy()
    pr_boxes = [] if pr_f is None else pr_f[["X", "Y", "Width", "Height"]].to_numpy()

    if len(gt_ids) == 0 and len(pr_ids) == 0:
        acc.update([], [], np.empty((0, 0)))
        continue

    # IoU cost matrix (motmetrics expects a COST: 1 - IoU)
    C = mm.distances.iou_matrix(gt_boxes, pr_boxes, max_iou=iou_threshold)

    acc.update(gt_ids, pr_ids, C)

# compute and print metrics
mh = mm.metrics.create()

summary = mh.compute(
    acc,
    metrics=[
        "num_frames",
        "mota",
        "motp",
        "idf1",
        "idp",
        "idr",
        "num_switches",
        "num_false_positives",
        "num_misses",
        "num_detections",
        "num_objects",
    ],
    name="OVERALL",
)

print("\n=== MOT evaluation ===\n")
print(mm.io.render_summary(summary, formatters=mh.formatters, namemap=mm.io.motchallenge_metric_names,))

FileNotFoundError: [Errno 2] No such file or directory: './gt/video7_gt.txt'